### Célula 1 — Lendo a Silver para o EDA

In [0]:
SILVER_PATH = "/Volumes/workspace/default/raw/silver/"        # caminho da Silver

df = spark.read.format("delta").load(SILVER_PATH)             # lê o dado limpo da Silver

# Visão geral
print(f"Linhas: {df.count()}")                                # total de registros
print(f"Colunas: {len(df.columns)}")                          # total de colunas
print()
df.printSchema()                                              # tipos de cada coluna

Linhas: 8469
Colunas: 18

root
 |-- Ticket_ID: integer (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Customer_Email: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Customer_Gender: string (nullable = true)
 |-- Product_Purchased: string (nullable = true)
 |-- Date_of_Purchase: date (nullable = true)
 |-- Ticket_Type: string (nullable = true)
 |-- Ticket_Subject: string (nullable = true)
 |-- Ticket_Description: string (nullable = true)
 |-- Ticket_Status: string (nullable = true)
 |-- Resolution: string (nullable = true)
 |-- Ticket_Priority: string (nullable = true)
 |-- Ticket_Channel: string (nullable = true)
 |-- First_Response_Time: timestamp (nullable = true)
 |-- Time_to_Resolution: timestamp (nullable = true)
 |-- Customer_Satisfaction_Rating: double (nullable = true)
 |-- _loaded_at: timestamp (nullable = true)



### Célula 2 — Estatísticas das colunas numéricas

In [0]:
# Estatísticas descritivas das colunas numéricas
df.select(
    "Customer_Age",                   # idade do cliente
    "Customer_Satisfaction_Rating"    # avaliação de satisfação
).describe().show()                   # count, mean, stddev, min, max

+-------+-----------------+----------------------------+
|summary|     Customer_Age|Customer_Satisfaction_Rating|
+-------+-----------------+----------------------------+
|  count|             8469|                        2769|
|   mean|44.02680363679301|           2.991332611050921|
| stddev|15.29611249888238|           1.407015861799693|
|    min|               18|                         1.0|
|    max|               70|                         5.0|
+-------+-----------------+----------------------------+



Estatísticas Numéricas — Insights

| Coluna | Insight |
|--------|---------|
| `Customer_Age` | Média 44 anos, range 18-70 — público adulto amplo |
| `Customer_Satisfaction_Rating` | Média **2.99** — abaixo de 3, preocupante! |
| `Customer_Satisfaction_Rating` | Só **2.769 avaliações** de 8.469 — 67% sem avaliação (tickets abertos) |

### Célula 3 — Distribuição por variáveis categóricas

In [0]:
from pyspark.sql.functions import count, round, col       # funções de agregação

# Função auxiliar para contar e calcular percentual
def distribuicao(coluna):
    total = df.count()                                    # total de registros
    return (df.groupBy(coluna)
        .agg(count("*").alias("quantidade"))              # conta por categoria
        .withColumn("percentual",                         # calcula percentual
            round((col("quantidade") / total) * 100, 1))
        .orderBy("quantidade", ascending=False)           # ordena do maior para menor
    )

# Distribuições
print("=== Ticket Status ===")
distribuicao("Ticket_Status").show()

print("=== Ticket Priority ===")
distribuicao("Ticket_Priority").show()

print("=== Ticket Channel ===")
distribuicao("Ticket_Channel").show()

print("=== Ticket Type ===")
distribuicao("Ticket_Type").show()

=== Ticket Status ===
+--------------------+----------+----------+
|       Ticket_Status|quantidade|percentual|
+--------------------+----------+----------+
|Pending Customer ...|      2881|      34.0|
|                Open|      2819|      33.3|
|              Closed|      2769|      32.7|
+--------------------+----------+----------+

=== Ticket Priority ===
+---------------+----------+----------+
|Ticket_Priority|quantidade|percentual|
+---------------+----------+----------+
|         Medium|      2192|      25.9|
|       Critical|      2129|      25.1|
|           High|      2085|      24.6|
|            Low|      2063|      24.4|
+---------------+----------+----------+

=== Ticket Channel ===
+--------------+----------+----------+
|Ticket_Channel|quantidade|percentual|
+--------------+----------+----------+
|         Email|      2143|      25.3|
|         Phone|      2132|      25.2|
|  Social media|      2121|      25.0|
|          Chat|      2073|      24.5|
+--------------+-----

#### Distribuição Categórica — Insights

#### Ticket Status
- Distribuição **quase uniforme** entre os 3 status (~33% cada)
- 67% dos tickets ainda **abertos ou pendentes** — explica os nulos em Satisfaction e Resolution

#### Ticket Priority
- Distribuição **surpreendentemente uniforme** (~25% cada)
- Não há concentração em prioridade baixa — volume alto de Critical e High é preocupante

#### Ticket Channel
- Todos os canais com volume **praticamente igual** (~25%)
- Nenhum canal dominante — clientes usam todos os canais igualmente

#### Ticket Type
- Distribuição também **uniforme** (~20% cada)
- **Refund request** e **Technical issue** lideram levemente
- Alta proporção de Cancellation request (20%) — sinal de alerta para o negócio

### Célula 4 — Satisfação média por categoria

In [0]:
from pyspark.sql.functions import avg, round, col

# Filtra apenas tickets com avaliação
df_sat = df.filter(col("Customer_Satisfaction_Rating").isNotNull())

# Satisfação por canal
print("=== Satisfação por Canal ===")
(df_sat.groupBy("Ticket_Channel")
    .agg(round(avg("Customer_Satisfaction_Rating"), 2).alias("satisfacao_media"))
    .orderBy("satisfacao_media", ascending=False)
    .show()
)

# Satisfação por prioridade
print("=== Satisfação por Prioridade ===")
(df_sat.groupBy("Ticket_Priority")
    .agg(round(avg("Customer_Satisfaction_Rating"), 2).alias("satisfacao_media"))
    .orderBy("satisfacao_media", ascending=False)
    .show()
)

# Satisfação por tipo
print("=== Satisfação por Tipo ===")
(df_sat.groupBy("Ticket_Type")
    .agg(round(avg("Customer_Satisfaction_Rating"), 2).alias("satisfacao_media"))
    .orderBy("satisfacao_media", ascending=False)
    .show()
)

=== Satisfação por Canal ===
+--------------+----------------+
|Ticket_Channel|satisfacao_media|
+--------------+----------------+
|          Chat|            3.08|
|  Social media|            2.97|
|         Email|            2.96|
|         Phone|            2.95|
+--------------+----------------+

=== Satisfação por Prioridade ===
+---------------+----------------+
|Ticket_Priority|satisfacao_media|
+---------------+----------------+
|            Low|            3.05|
|           High|            2.98|
|         Medium|            2.98|
|       Critical|            2.96|
+---------------+----------------+

=== Satisfação por Tipo ===
+--------------------+----------------+
|         Ticket_Type|satisfacao_media|
+--------------------+----------------+
|     Billing inquiry|            3.03|
|Cancellation request|            3.03|
|     Product inquiry|            3.02|
|     Technical issue|            2.96|
|      Refund request|            2.93|
+--------------------+-------------

#### Satisfação por Categoria — Insights

#### Satisfação Geral: 2.99 ⚠️
- Média **abaixo de 3.0** — sinal vermelho para o negócio
- Apenas **2.769 avaliações** de 8.469 tickets — 67% sem avaliação (tickets ainda abertos)

#### Por Canal
| Canal | Satisfação | Insight |
|-------|-----------|---------|
| Chat | 3.08 ✅ | Único acima da média — tempo real agrada |
| Social media | 2.97 ⚠️ | Abaixo da média |
| Email | 2.96 ⚠️ | Abaixo da média |
| Phone | 2.95 ⚠️ | Pior canal — espera e transferências frustram |

#### Por Prioridade
| Prioridade | Satisfação | Insight |
|------------|-----------|---------|
| Low | 3.05 ✅ | Única acima da média |
| High | 2.98 ⚠️ | Abaixo da média |
| Medium | 2.98 ⚠️ | Abaixo da média |
| Critical | 2.96 ⚠️ | Pior — tickets urgentes não estão sendo bem resolvidos |

#### Por Tipo
| Tipo | Satisfação | Insight |
|------|-----------|---------|
| Billing inquiry | 3.03 ✅ | Acima da média |
| Cancellation request | 3.03 ✅ | Acima da média — surpreendente |
| Product inquiry | 3.02 ✅ | Acima da média |
| Technical issue | 2.96 ⚠️ | Abaixo da média |
| Refund request | 2.93 ⚠️ | Pior tipo — cliente já frustrado ao abrir o ticket |

### Célula 5 — Volume de tickets por período

In [0]:
from pyspark.sql.functions import year, month, count    # funções de data e agregação

# Volume por ano
print("=== Tickets por Ano ===")
(df.groupBy(year("Date_of_Purchase").alias("ano"))      # extrai o ano da data
    .agg(count("*").alias("quantidade"))                # conta tickets por ano
    .orderBy("ano")                                     # ordena cronologicamente
    .show()
)

# Volume por mês
print("=== Tickets por Mês ===")
(df.groupBy(month("Date_of_Purchase").alias("mes"))     # extrai o mês da data
    .agg(count("*").alias("quantidade"))                # conta tickets por mês
    .orderBy("mes")                                     # ordena cronologicamente
    .show()
)

=== Tickets por Ano ===
+----+----------+
| ano|quantidade|
+----+----------+
|2020|      4236|
|2021|      4233|
+----+----------+

=== Tickets por Mês ===
+---+----------+
|mes|quantidade|
+---+----------+
|  1|       736|
|  2|       715|
|  3|       672|
|  4|       718|
|  5|       701|
|  6|       678|
|  7|       727|
|  8|       691|
|  9|       696|
| 10|       735|
| 11|       704|
| 12|       696|
+---+----------+



#### Análise Temporal — Insights

#### Volume por Ano
| Ano | Tickets | Insight |
|-----|---------|---------|
| 2020 | 4.236 | Volume levemente maior |
| 2021 | 4.233 | Volume estável — queda insignificante |

- Distribuição **praticamente igual** entre os dois anos
- Sem crescimento nem queda relevante — volume estável

#### Volume por Mês
- Distribuição **uniforme** ao longo do ano — sem sazonalidade clara
- Pico em **Janeiro (736)** e **Outubro (735)**
- Vale em **Março (672)** e **Junho (678)**
- Variação máxima entre meses: **64 tickets** — irrelevante estatisticamente

#### Conclusão
> O volume de tickets é **previsível e estável** — sem picos sazonais.
> Isso facilita o planejamento de capacidade da equipe de suporte.

### Célula 6 — Produtos com mais tickets

In [0]:
# Top 10 produtos com mais tickets
print("=== Top 10 Produtos com mais Tickets ===")
(df.groupBy("Product_Purchased")
    .agg(count("*").alias("quantidade"))                # conta tickets por produto
    .orderBy("quantidade", ascending=False)             # ordena do maior para menor
    .limit(10)                                          # top 10 apenas
    .show()
)

=== Top 10 Produtos com mais Tickets ===
+-------------------+----------+
|  Product_Purchased|quantidade|
+-------------------+----------+
|          Canon EOS|       240|
|         GoPro Hero|       228|
|    Nest Thermostat|       225|
|        Amazon Echo|       221|
| Philips Hue Lights|       221|
|        LG Smart TV|       219|
|        Sony Xperia|       217|
|Roomba Robot Vacuum|       216|
|      Apple AirPods|       213|
|            LG OLED|       213|
+-------------------+----------+



#### Top 10 Produtos com mais Tickets — Insights

| Produto | Tickets | Insight |
|---------|---------|---------|
| Canon EOS | 240 | Líder em volume de tickets |
| GoPro Hero | 228 | Segundo maior — produtos de câmera lideram |
| Nest Thermostat | 225 | IoT/Smart Home com alto volume |
| Amazon Echo | 221 | Assistente virtual gera muitos chamados |
| Philips Hue Lights | 221 | Smart Home novamente |

#### Conclusão
- **Câmeras** (Canon EOS, GoPro Hero) lideram os chamados — produtos com setup complexo
- **Smart Home** (Nest, Echo, Philips Hue) aparecem fortemente — integração e configuração geram dúvidas
- Distribuição **relativamente uniforme** — sem um produto dominante isolado
- Todos os 42 produtos têm volume médio de ~200 tickets — base bem distribuída

### Célula 7 — Taxa de resolução

In [0]:
import pyspark.sql.functions as F
import builtins

total = df.count()

resolvidos = df.filter(
    F.col("Ticket_Status") == "Closed"
).count()

print(f"Total de tickets:   {total}")
print(f"Tickets resolvidos: {resolvidos}")

print(
    f"Taxa de resolução: "
    f"{builtins.round(resolvidos/total*100, 1)}%"
)

df.groupBy("Ticket_Priority") \
    .agg(
        F.count("*").alias("total"),
        F.count(
            F.when(
                F.col("Ticket_Status") == "Closed",
                1
            )
        ).alias("resolvidos")
    ) \
    .orderBy(F.desc("resolvidos")) \
    .show()

Total de tickets:   8469
Tickets resolvidos: 2769
Taxa de resolução: 32.7%
+---------------+-----+----------+
|Ticket_Priority|total|resolvidos|
+---------------+-----+----------+
|       Critical| 2129|       726|
|           High| 2085|       705|
|         Medium| 2192|       694|
|            Low| 2063|       644|
+---------------+-----+----------+



#### Taxa de Resolução — Insights

#### Geral
- Apenas **32.7%** dos tickets estão resolvidos (Closed)
- **67.3%** ainda abertos ou pendentes — volume alto de backlog

#### Por Prioridade
| Prioridade | Total | Resolvidos | Taxa |
|------------|-------|------------|------|
| Critical | 2.129 | 726 | 34.1% |
| High | 2.085 | 705 | 33.8% |
| Medium | 2.192 | 694 | 31.7% |
| Low | 2.063 | 644 | 31.2% |

#### Conclusão
- Taxas de resolução **muito próximas** entre prioridades (~32%)
- **Critical resolve ligeiramente mais** — mas a diferença é pequena
- Esperava-se que Low tivesse maior taxa — na prática a prioridade **não está impactando** a resolução
- Isso é um insight importante para o negócio — o SLA por prioridade pode não estar sendo respeitado

## Salvando Resumo do EDA — Delta Lake

- Filtra apenas tickets com avaliação de satisfação
- Agrupa por **Ticket_Type**, **Ticket_Channel** e **Ticket_Priority**
- Calcula **total de tickets** e **satisfação média** por combinação
- Salvo em `/Volumes/workspace/default/raw/eda/` como Delta Lake
- Serve como referência para consultas futuras sem reprocessar o EDA

In [0]:
import pyspark.sql.functions as F

# Resumo de satisfação por categoria para referência futura
resumo_eda = df.filter(F.col("Customer_Satisfaction_Rating").isNotNull()) \
    .groupBy("Ticket_Type", "Ticket_Channel", "Ticket_Priority") \
    .agg(
        F.count("*").alias("total_tickets"),
        F.round(F.avg("Customer_Satisfaction_Rating"), 2).alias("satisfacao_media")
    )

EDA_PATH = "/Volumes/workspace/default/raw/eda/"
resumo_eda.write.format("delta").mode("overwrite").save(EDA_PATH)
print("✅ Resumo EDA salvo!")

✅ Resumo EDA salvo!
